# 04 - Data Quality Checks

## Purpose

Run rule-driven data quality checks against Silver and Gold tables before reporting consumption.

## Concepts covered

- Rule configuration from YAML
- Not-null checks
- Duplicate key checks
- Accepted values checks
- Numeric range checks
- Referential integrity checks
- Row count reconciliation
- Freshness checks
- Markdown-style DQ reporting

## Prerequisites

- 02_silver_transformation.ipynb completed successfully.
- 03_gold_dimensional_model.ipynb completed successfully if Gold rules are enabled.
- Optional: upload data-quality/dq_rules.yml and data-quality/dq_framework.py to Files/retail_banking/config.

## Expected output

A displayed DQ results table, a persisted dq_results Delta table, and a markdown-style quality report.

In [ ]:
from datetime import datetime, timezone
import os
import sys
from typing import Any, Dict, List

from pyspark.sql import functions as F

CONFIG_PATHS = [
    "/lakehouse/default/Files/retail_banking/config",
    "Files/retail_banking/config",
    "data-quality",
    "../data-quality",
]
RULE_FILE_NAME = "dq_rules.yml"
RESULTS_TABLE = "dq_results"

for config_path in CONFIG_PATHS:
    if os.path.isdir(config_path) and config_path not in sys.path:
        sys.path.append(config_path)

def log_info(message: str) -> None:
    print(f"[INFO] {datetime.now(timezone.utc).isoformat()} | {message}")

def log_warning(message: str) -> None:
    print(f"[WARN] {datetime.now(timezone.utc).isoformat()} | {message}")

## Load YAML rules

The preferred pattern is to keep rules in data-quality/dq_rules.yml. The embedded fallback keeps this notebook runnable during demos before the config file is uploaded to the Lakehouse.

In [ ]:
DEFAULT_RULES: Dict[str, Any] = {
    "settings": {"fail_on_critical": True, "results_table": RESULTS_TABLE},
    "tables": {
        "silver_customers": {"rules": [
            {"name": "customer_id_required", "type": "not_null", "columns": ["customer_id"], "severity": "critical"},
            {"name": "customer_id_unique", "type": "duplicate_key", "columns": ["customer_id"], "severity": "critical"},
            {"name": "customer_status_valid", "type": "accepted_values", "column": "customer_status", "values": ["Active", "Inactive"], "severity": "critical"},
        ]},
        "silver_accounts": {"rules": [
            {"name": "account_id_required", "type": "not_null", "columns": ["account_id", "customer_id", "product_id", "branch_id"], "severity": "critical"},
            {"name": "account_id_unique", "type": "duplicate_key", "columns": ["account_id"], "severity": "critical"},
            {"name": "account_status_valid", "type": "accepted_values", "column": "account_status", "values": ["Open", "Closed"], "severity": "critical"},
            {"name": "account_balance_reasonable", "type": "range", "column": "current_balance", "min": -1000000, "max": 10000000, "severity": "warning"},
            {"name": "account_customer_integrity", "type": "referential_integrity", "column": "customer_id", "parent_table": "silver_customers", "parent_column": "customer_id", "severity": "critical"},
        ]},
        "silver_transactions": {"rules": [
            {"name": "transaction_id_required", "type": "not_null", "columns": ["transaction_id", "account_id", "transaction_timestamp", "transaction_amount"], "severity": "critical"},
            {"name": "transaction_id_unique", "type": "duplicate_key", "columns": ["transaction_id"], "severity": "critical"},
            {"name": "transaction_status_valid", "type": "accepted_values", "column": "transaction_status", "values": ["Posted", "Rejected", "Pending"], "severity": "critical"},
            {"name": "transaction_account_integrity", "type": "referential_integrity", "column": "account_id", "parent_table": "silver_accounts", "parent_column": "account_id", "severity": "critical"},
            {"name": "transaction_freshness", "type": "freshness", "column": "_ingestion_timestamp", "max_age_hours": 72, "severity": "warning"},
        ]},
        "fact_transaction": {"rules": [
            {"name": "fact_has_rows", "type": "row_count_min", "min_count": 1, "severity": "critical"},
            {"name": "fact_transaction_id_unique", "type": "duplicate_key", "columns": ["transaction_id"], "severity": "critical"},
            {"name": "silver_to_gold_transaction_reconciliation", "type": "row_count_reconciliation", "source_table": "silver_transactions", "target_table": "fact_transaction", "severity": "critical"},
        ]},
    },
}

def read_rules_text() -> str:
    candidates = [os.path.join(base_path, RULE_FILE_NAME) for base_path in CONFIG_PATHS]
    candidates.append(RULE_FILE_NAME)
    for candidate in candidates:
        if os.path.exists(candidate):
            log_info(f"Loading DQ rules from {candidate}")
            with open(candidate, "r", encoding="utf-8") as handle:
                return handle.read()
    raise FileNotFoundError("Could not find dq_rules.yml in configured paths")

try:
    import yaml
    rules = yaml.safe_load(read_rules_text())
except Exception as exc:
    log_warning(f"Using embedded default DQ rules because YAML rules could not be loaded: {exc}")
    rules = DEFAULT_RULES

rules_summary_df = spark.createDataFrame([
    {"config_section": "tables", "configured_count": len(rules.get("tables", {}))},
    {"config_section": "settings", "configured_count": len(rules.get("settings", {}))},
])
display(rules_summary_df)

## Import the reusable framework

The repository includes data-quality/dq_framework.py. When that file is available in the notebook path, this notebook imports it. Otherwise it uses a small fallback implementation with the same core behavior.

In [ ]:
try:
    from dq_framework import DataQualityRunner
    log_info("Imported DataQualityRunner from dq_framework.py")
except Exception as exc:
    log_warning(f"Could not import dq_framework.py. Using notebook fallback runner. Reason: {exc}")

    class DataQualityRunner:
        def __init__(self, spark_session, rule_config):
            self.spark = spark_session
            self.rules = rule_config
            self.run_timestamp = datetime.now(timezone.utc).isoformat()
            self.results: List[Dict[str, Any]] = []

        def _add(self, table_name, rule, status, failed_count, details):
            self.results.append({
                "table_name": table_name,
                "rule_name": rule.get("name", rule.get("type", "unknown")),
                "rule_type": rule.get("type", "unknown"),
                "severity": rule.get("severity", "warning"),
                "status": status,
                "failed_count": int(failed_count),
                "details": details,
                "run_timestamp": self.run_timestamp,
            })

        def _table(self, table_name):
            if not self.spark.catalog.tableExists(table_name):
                raise RuntimeError(f"Table does not exist: {table_name}")
            return self.spark.table(table_name)

        def run(self):
            for table_name, table_config in self.rules.get("tables", {}).items():
                for rule in table_config.get("rules", []):
                    try:
                        self._run_rule(table_name, rule)
                    except Exception as error:
                        self._add(table_name, rule, "FAIL", 1, str(error)[:500])
            return self.to_dataframe()

        def to_dataframe(self):
            schema = "table_name string, rule_name string, rule_type string, severity string, status string, failed_count long, details string, run_timestamp string"
            return self.spark.createDataFrame(self.results, schema=schema) if self.results else self.spark.createDataFrame([], schema=schema)

        def markdown_report(self):
            lines = ["# Data Quality Report", "", f"Run timestamp: {self.run_timestamp}", "", "| Table | Rule | Type | Severity | Status | Failed Count | Details |", "| --- | --- | --- | --- | --- | ---: | --- |"]
            for row in self.results:
                lines.append(f"| {row['table_name']} | {row['rule_name']} | {row['rule_type']} | {row['severity']} | {row['status']} | {row['failed_count']} | {row['details']} |")
            return "\n".join(lines)

        def _run_rule(self, table_name, rule):
            rule_type = rule.get("type")
            df = self._table(table_name)
            if rule_type == "not_null":
                for column_name in rule.get("columns", []):
                    failed = df.filter(F.col(column_name).isNull() | (F.trim(F.col(column_name).cast("string")) == "")).count()
                    child_rule = dict(rule, name=f"{rule.get('name')}:{column_name}")
                    self._add(table_name, child_rule, "PASS" if failed == 0 else "FAIL", failed, f"{column_name} must not be null or blank")
            elif rule_type == "duplicate_key":
                columns = rule.get("columns", [])
                failed = df.groupBy(*columns).count().filter(F.col("count") > 1).count()
                self._add(table_name, rule, "PASS" if failed == 0 else "FAIL", failed, f"Unique key: {columns}")
            elif rule_type == "accepted_values":
                failed = df.filter(~F.col(rule["column"]).isin(rule.get("values", [])) | F.col(rule["column"]).isNull()).count()
                self._add(table_name, rule, "PASS" if failed == 0 else "FAIL", failed, f"Accepted values: {rule.get('values', [])}")
            elif rule_type == "range":
                condition = F.lit(False)
                if "min" in rule:
                    condition = condition | (F.col(rule["column"]) < F.lit(rule["min"]))
                if "max" in rule:
                    condition = condition | (F.col(rule["column"]) > F.lit(rule["max"]))
                failed = df.filter(condition).count()
                self._add(table_name, rule, "PASS" if failed == 0 else "FAIL", failed, f"Range {rule.get('min')} to {rule.get('max')}")
            elif rule_type == "referential_integrity":
                child = df.filter(F.col(rule["column"]).isNotNull()).select(rule["column"]).distinct().alias("child")
                parent = self._table(rule["parent_table"]).select(rule["parent_column"]).distinct().alias("parent")
                failed = child.join(parent, F.col(f"child.{rule['column']}") == F.col(f"parent.{rule['parent_column']}"), "left_anti").count()
                self._add(table_name, rule, "PASS" if failed == 0 else "FAIL", failed, f"{rule['column']} must exist in {rule['parent_table']}.{rule['parent_column']}")
            elif rule_type == "row_count_min":
                actual = df.count()
                minimum = int(rule.get("min_count", 1))
                failed = 0 if actual >= minimum else minimum - actual
                self._add(table_name, rule, "PASS" if failed == 0 else "FAIL", failed, f"Actual={actual}, minimum={minimum}")
            elif rule_type == "row_count_reconciliation":
                source_count = self._table(rule["source_table"]).count()
                target_count = self._table(rule["target_table"]).count()
                failed = abs(source_count - target_count)
                self._add(table_name, rule, "PASS" if failed == 0 else "FAIL", failed, f"Source={source_count}, target={target_count}")
            elif rule_type == "freshness":
                row = df.agg(((F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp(F.max(F.col(rule["column"])))) / 3600).alias("age_hours")).collect()[0]
                age_hours = row["age_hours"]
                max_age = int(rule.get("max_age_hours", 24))
                failed = 0 if age_hours is not None and age_hours <= max_age else 1
                self._add(table_name, rule, "PASS" if failed == 0 else "FAIL", failed, f"Age hours={age_hours}, max={max_age}")
            else:
                self._add(table_name, rule, "FAIL", 1, f"Unsupported rule type: {rule_type}")

## Run DQ checks

In [ ]:
runner = DataQualityRunner(spark, rules)
dq_results_df = runner.run().orderBy("severity", "table_name", "rule_name")

display(dq_results_df)

target_results_table = rules.get("settings", {}).get("results_table", RESULTS_TABLE)
dq_results_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_results_table)

print(runner.markdown_report())

## Fail the pipeline on critical failures

In [ ]:
summary_df = (
    dq_results_df
    .groupBy("severity", "status")
    .agg(F.count("*").alias("rule_count"), F.sum("failed_count").alias("failed_row_count"))
    .orderBy("severity", "status")
)
display(summary_df)

critical_failure_count = dq_results_df.filter((F.col("severity") == "critical") & (F.col("status") == "FAIL")).count()
fail_on_critical = bool(rules.get("settings", {}).get("fail_on_critical", True))

if fail_on_critical and critical_failure_count > 0:
    raise RuntimeError(f"Data quality failed with {critical_failure_count} critical rule failures")

log_info("Data quality checks completed successfully")

## Next steps

Run 05_delta_optimization.ipynb to inspect table health and apply safe Delta maintenance patterns.